# Full Resource Inventory + Godot Report Review
One code cell performs the full audit, checks the prior Godot result, inventories the complete original archive, and uploads verified outputs to the existing Release.

In [ ]:
#@title Full Resource Inventory + Godot Report Review (single cell)
import os, re, json, csv, time, hashlib, zipfile, shutil, pathlib, requests, collections, mimetypes
from IPython.display import display, Markdown

OWNER="ZzwahwahzZ"; ASSET_REPO="Conquer-Mobile-Assets."; TAG="mobile-extracted-assets"
WORK=pathlib.Path("/content/clash_asset_full_audit"); shutil.rmtree(WORK,ignore_errors=True); WORK.mkdir(parents=True)
TOKEN=None
try:
    from google.colab import userdata
    TOKEN=userdata.get("GITHUB_TOKEN")
except Exception: pass
if not TOKEN: raise RuntimeError("Missing Colab Secret GITHUB_TOKEN. Add it, then rerun this single cell.")
H={"Authorization":f"Bearer {TOKEN}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
API=f"https://api.github.com/repos/{OWNER}/{ASSET_REPO}"
def release_asset(name):
    r=requests.get(f"{API}/releases/tags/{TAG}",headers=H,timeout=60); r.raise_for_status()
    items=r.json()["assets"]
    a=next((x for x in items if x["name"]==name),None)
    if not a: raise RuntimeError(f"Required release asset missing: {name}")
    return a
def download_asset(name,dest):
    a=release_asset(name); dest=pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size==a["size"]: return dest
    u=a["browser_download_url"]; print("Downloading",name,"(",a["size"],"bytes )")
    with requests.get(u,headers={"Authorization":f"Bearer {TOKEN}"},stream=True,timeout=(30,180),allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest,"wb") as f:
            for chunk in r.iter_content(8*1024*1024):
                if chunk: f.write(chunk)
    if dest.stat().st_size!=a["size"]: raise RuntimeError(f"Size mismatch for {name}: {dest.stat().st_size} != {a['size']}")
    print("VERIFIED DOWNLOAD",name,dest.stat().st_size)
    return dest
# 1) Review the exact Godot report and log produced by the completed visual-validation run.
validation_zip=download_asset("conquer-mobile-visual-validation.zip",WORK/"visual-validation.zip")
with zipfile.ZipFile(validation_zip) as z:
    bad=z.testzip()
    if bad: raise RuntimeError("Validation ZIP CRC failure: "+bad)
    names=z.namelist()
    report_member=next((n for n in names if n.endswith("godot-4.3-import-report.json")),None)
    log_member=next((n for n in names if n.endswith("godot-4.3-import-log.txt")),None)
    if not report_member: raise RuntimeError("Godot import report is missing from validation ZIP")
    report=json.loads(z.read(report_member).decode("utf-8"))
    log=z.read(log_member).decode("utf-8",errors="replace") if log_member else ""
print("\n=== GODOT 4.3 IMPORT RESULT ===")
print(json.dumps(report,ensure_ascii=False,indent=2))
print("=== GODOT LOG ===")
print(log[:12000] if log else "(empty log)")
# 2) Download and safely inventory every file in the original extracted-asset archive.
source_zip=download_asset("extracted-assets-001.zip",WORK/"extracted-assets-001.zip")
ROOT=WORK/"source"; ROOT.mkdir()
rows=[]; seen=set(); totals=collections.Counter(); ext_counts=collections.Counter()
MODEL={".glb",".gltf",".obj",".fbx",".dae",".blend",".3ds",".stl",".ply",".mtl"}
IMAGE={".png",".jpg",".jpeg",".webp",".tga",".bmp",".gif",".dds",".ktx",".ktx2",".psd"}
AUDIO={".ogg",".mp3",".wav",".flac",".m4a",".aac",".opus",".mid",".midi",".wma",".aiff",".xm",".mod",".it",".s3m"}
VIDEO={".mp4",".webm",".mov",".avi",".mkv"}
FONT={".ttf",".otf",".woff",".woff2"}
with zipfile.ZipFile(source_zip) as z:
    bad=z.testzip()
    if bad: raise RuntimeError("Source ZIP CRC failure: "+bad)
    infos=[i for i in z.infolist() if not i.is_dir()]
    print("\nArchive entries:",len(infos))
    for i,info in enumerate(infos,1):
        p=pathlib.PurePosixPath(info.filename)
        if p.is_absolute() or ".." in p.parts: raise RuntimeError("Unsafe archive path: "+info.filename)
        name=str(p); ext=p.suffix.lower()
        if name in seen: raise RuntimeError("Duplicate archive path: "+name)
        seen.add(name)
        low=name.lower()
        if ext in MODEL: cat="3d_model"
        elif ext in IMAGE: cat="image_texture"
        elif ext in AUDIO: cat="audio"
        elif ext in VIDEO: cat="video"
        elif ext in FONT: cat="font"
        elif any(k in low for k in ("background","bg_","loading","login","interface","ui/","icon","button","skill","inventory","hud","map/","scene/")): cat="ui_or_background_candidate"
        elif ext in {".json",".xml",".txt",".csv",".ini",".cfg",".dat",".bin",".db",".bytes",".atlas",".plist",".fnt",".shader",".gdshader"}: cat="game_data_or_config"
        else: cat="other"
        totals[cat]+=1; ext_counts[ext or "[no extension]"]+=1
        rows.append({"path":name,"category":cat,"extension":ext,"bytes":info.file_size,"compressed_bytes":info.compress_size})
        if i%50==0: print(f"Indexed {i}/{len(infos)}")
    # Extract only supplemental categories; original archive is CRC-checked above.
    supplemental=WORK/"supplemental"; supplemental.mkdir()
    chosen=[r for r in rows if r["category"] not in ("3d_model","image_texture")]
    for r in chosen:
        dest=supplemental/r["path"]; dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(r["path"]) as src, open(dest,"wb") as out: shutil.copyfileobj(src,out)
# 3) Create auditable inventories and a supplemental pack without altering the game project.
with open(WORK/"complete-asset-inventory.csv","w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=["path","category","extension","bytes","compressed_bytes"]); w.writeheader(); w.writerows(rows)
summary={"source_archive":"extracted-assets-001.zip","archive_entries":len(rows),"category_counts":dict(totals),"extension_counts":dict(ext_counts),"godot_import_report":report,"godot_log_excerpt":log[:12000],"limitations":["This is an inventory and isolated import report review; it does not validate Android runtime performance.","UI/background candidates are filename/path heuristics and require visual confirmation.","No files in the actual Clash-of-Legends game project were modified."]}
(WORK/"complete-asset-inventory.json").write_text(json.dumps({"summary":summary,"files":rows},ensure_ascii=False,indent=2),encoding="utf-8")
(WORK/"asset-category-summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
supp_zip=WORK/"conquer-mobile-supplemental-assets.zip"
with zipfile.ZipFile(supp_zip,"w",zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in supplemental.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(supplemental).as_posix())
print("\nCATEGORY COUNTS")
for k,v in totals.most_common(): print(f"{k}: {v}")
print("SUPPLEMENTAL FILES:",len(chosen),"PACKAGE BYTES:",supp_zip.stat().st_size)
# 4) Upload inventory reports and supplemental pack to the existing release, then verify each asset.
def upload(path,name):
    old=next((a for a in release_asset_list if a["name"]==name),None)
    if old:
        # Never silently replace a published asset; use a versioned name if already present.
        raise RuntimeError("Release already contains "+name+"; refusing to overwrite.")
    u=f"https://uploads.github.com/repos/{OWNER}/{ASSET_REPO}/releases/{release_id}/assets?name={name}"
    with open(path,"rb") as f:
        r=requests.post(u,headers={**H,"Content-Type":"application/octet-stream"},data=f,timeout=(30,600))
    r.raise_for_status(); a=r.json()
    if a.get("state")!="uploaded" or a.get("size")!=path.stat().st_size: raise RuntimeError("Upload verification failed: "+name)
    print("UPLOADED + VERIFIED",name,a["size"])
release=requests.get(f"{API}/releases/tags/{TAG}",headers=H,timeout=60); release.raise_for_status()
release_data=release.json(); release_id=release_data["id"]; release_asset_list=release_data["assets"]
for p,n in [(WORK/"complete-asset-inventory.csv","complete-asset-inventory.csv"),(WORK/"complete-asset-inventory.json","complete-asset-inventory.json"),(WORK/"asset-category-summary.json","asset-category-summary.json"),(supp_zip.name and supp_zip,"conquer-mobile-supplemental-assets.zip")]:
    upload(p,n)
print("\nSUCCESS: full inventory, Godot report review, supplemental asset package uploaded and verified.")
print("Release:",release_data["html_url"])
display(Markdown("### Completed\n- Reviewed Godot 4.3 report/log from the validation bundle.\n- Indexed every original archive entry and classified audio, images, 3D, UI/background candidates, fonts, video, data, and other files.\n- Packaged all non-model/non-image resources separately.\n- Uploaded and verified the inventory reports and supplemental package.\n\n**No actual game project files were changed.**"))
